# Clase 09 · Codificación de variables y escalamiento — práctica guiada

Recorremos la clase paso a paso con `data/raw/viviendas_lima.csv` (el profesor usó el dataset *bank marketing*;
aquí usamos los datasets del repo, que tienen las mismas situaciones).

**Ruta:** 1) cada fila es un punto → 2) separar numéricas / categóricas → 3) analizar contra el objetivo →
4) encoders (OneHot, dummy, ordinal, binario) → 5) transformar (PowerTransformer) vs escalar (MinMax/Standard) →
6) normalizar por fila → 7) todo junto en `Pipeline` + `ColumnTransformer`.

> Nota: en este entorno no hay matplotlib, así que miramos las distribuciones con números (`describe`, `skew`).

In [ ]:
import math
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (OneHotEncoder, OrdinalEncoder, PowerTransformer,
                                   StandardScaler, MinMaxScaler, Normalizer)

# Busca la carpeta data/raw subiendo desde la carpeta actual
raiz = Path.cwd()
while not (raiz / "data" / "raw").exists() and raiz != raiz.parent:
    raiz = raiz.parent
viv = pd.read_csv(raiz / "data" / "raw" / "viviendas_lima.csv")
banco = pd.read_csv(raiz / "data" / "raw" / "clientes_banco.csv")
viv.head()

## 1. Cada fila es un punto en $\mathbb{R}^n$
*"Cada una de estas muestras representa un puntito"* en el **espacio de características**.
Un punto se describe con **coordenadas numéricas** $(x_1, x_2, \dots, x_n)$. Por eso:
- las **categóricas** (`distrito`, `estado`…) **deben codificarse** (no existe la coordenada "Miraflores");
- las **numéricas** deben estar en **escalas similares**. Mira las escalas de viviendas:

In [ ]:
viv[["habitaciones", "area_m2", "precio_usd"]].describe().loc[["min", "mean", "max"]]

`habitaciones` va de 0 a 7, `area_m2` de decenas a miles y `precio_usd` llega a millones.
Si calculamos distancias entre puntos (kNN, SVM, PCA…), **el precio dominaría todo** solo por su escala.

In [ ]:
# Distancia entre las dos primeras viviendas: ¿qué columna pesa más?
cols3 = ["habitaciones", "area_m2", "precio_usd"]
a, b = viv.loc[0, cols3], viv.loc[1, cols3]
print("Diferencias por columna:\n", (a - b).abs())
print("Distancia euclidiana:", round(float(np.sqrt(((a - b) ** 2).sum())), 1))

## 2. Separar numéricas y categóricas
Automático con `select_dtypes` (como en la clase). La variable objetivo (`precio_usd`) **no se transforma**
(y `id` tampoco es una característica).

In [ ]:
numericas = viv.select_dtypes(include="number").columns.drop(["precio_usd", "id"]).tolist()
categoricas = viv.select_dtypes(exclude="number").columns.tolist()
print("Numéricas:", numericas)
print("Categóricas:", categoricas)

El profesor recomienda **revisar a mano** después: ¿cuántas categorías tiene cada una?
Eso decide el encoder (2–3 → OneHot; muchas → binario u otro).

In [ ]:
viv[categoricas].nunique().sort_values()

`cochera` tiene 5 "categorías" porque el texto está sucio (`Si`, `si`, `SI`, `No`, `no `). **Primero se limpia** (clase 03),
después se codifica. El orden es **1) imputar nulos → 2) outliers → 3) encoding / transformación / escalamiento**.

## 3. Analizar contra la variable objetivo
En la clase usó **pivottablejs** (tablas dinámicas tipo Excel, exportables a HTML). Aquí hacemos lo mismo con
`pd.crosstab` / `pivot_table` (📌 Extra: pivottablejs no está instalado en este entorno).
En `clientes_banco.csv` el objetivo es `abandono`:

In [ ]:
print(banco["abandono"].value_counts())                  # ¿desbalanceado?
pd.crosstab(banco["segmento"], banco["abandono"], normalize="index").round(3)

In [ ]:
# En viviendas (regresión): precio mediano por estado y tipo de inmueble
viv.pivot_table(index="estado", columns="tipo_inmueble", values="precio_usd", aggfunc="median").round(-3)

## 4. Encoders para categóricas
### 4.1 OneHot (n columnas) vs dummy (n − 1 columnas)

In [ ]:
mini = pd.DataFrame({"team": ["A", "B", "C", "A"]})
print("OneHot (n columnas):")
print(pd.get_dummies(mini, columns=["team"], dtype=int))
print("\nDummy (n-1 columnas: 'A' queda como 0 0):")
print(pd.get_dummies(mini, columns=["team"], drop_first=True, dtype=int))

### 4.2 La maldición de la dimensionalidad
Ejemplo de la pregunta de Carlos: 10 variables, de las cuales 4 son categóricas con 4 categorías cada una.

In [ ]:
numericas_ej, n_cat, k = 6, 4, 4
print("OneHot :", numericas_ej + n_cat * k, "dimensiones")
print("Dummy  :", numericas_ej + n_cat * (k - 1), "dimensiones")
print("Binario:", numericas_ej + n_cat * math.ceil(math.log2(k)), "dimensiones")
print("distrito (15 categorías) -> OneHot 15, dummy 14, binario", math.ceil(math.log2(15)))

### 4.3 Ordinal: una sola columna… pero inventa un orden
`OrdinalEncoder` da 0, 1, 2… El modelo podría creer que un distrito "pesa más" que otro (los distritos no tienen orden).
A veces no afecta: **hay que probar y medir**.

In [ ]:
oe = OrdinalEncoder()
codigos = oe.fit_transform(viv[["distrito"]])
pd.DataFrame({"distrito": viv["distrito"], "codigo": codigos[:, 0]}).drop_duplicates().sort_values("codigo").head(6)

### 4.4 Binario: $\lceil \log_2 n \rceil$ columnas
A cada categoría se le da un código y se escribe en binario. Con 15 distritos bastan 4 bits.
La librería `category_encoders` tiene `BinaryEncoder` (con `inverse_transform`); aquí lo vemos "a mano" para entenderlo.

In [ ]:
cats = sorted(viv["distrito"].unique())
tabla = pd.DataFrame({"distrito": cats, "codigo": range(len(cats))})
tabla["binario"] = tabla["codigo"].apply(lambda c: format(c, "04b"))
tabla

Para un humano `0110` es difícil de leer; para la máquina es perfecto. Lo importante es que **dos categorías distintas
nunca tengan el mismo código**.

## 5. Transformar ≠ escalar
- **PowerTransformer** (Yeo-Johnson por defecto; Box-Cox solo con positivos) **cambia la forma** para acercarla a una normal.
  Tiene `inverse_transform`. No se aplica a `y`.
- **Escaladores** (`MinMaxScaler` → [0, 1]; `StandardScaler` → media 0, varianza 1) **no cambian la forma**, solo el rango.

Medimos la forma con la **asimetría** (`skew`): 0 = simétrica.

In [ ]:
precio = viv[["precio_usd"]]
comparacion = pd.DataFrame({
    "original": precio["precio_usd"],
    "minmax": MinMaxScaler().fit_transform(precio)[:, 0],
    "standard": StandardScaler().fit_transform(precio)[:, 0],
    "power (yeo-johnson)": PowerTransformer().fit_transform(precio)[:, 0],
})
pd.DataFrame({"skew": comparacion.skew(), "min": comparacion.min(), "max": comparacion.max()}).round(3)

👀 MinMax y Standard tienen **la misma asimetría** que el original; PowerTransformer la acerca a 0.

Modelos sensibles a escala / normalidad: regresión lineal y logística, SVM, kNN, PCA.
No les afecta: árboles, Random Forest, XGBoost, LightGBM.

## 6. Normalizar = por FILA
Cada punto se divide por su norma (L2: $\sqrt{x_1^2+\dots+x_n^2}$) y queda con longitud 1.
Esto **sí** cambia la distribución de cada variable.

In [ ]:
cols = ["area_m2", "habitaciones", "distancia_centro_km"]
datos = viv.dropna(subset=cols)[cols]
norm = pd.DataFrame(Normalizer(norm="l2").fit_transform(datos), columns=cols)
print("Norma de las 3 primeras filas:", np.sqrt((norm.head(3) ** 2).sum(axis=1)).round(6).tolist())
print("skew area_m2 antes:", round(datos["area_m2"].skew(), 3), "| después:", round(norm["area_m2"].skew(), 3))

## 7. Todo junto: `Pipeline` + `ColumnTransformer`
Un pipeline por tipo de columna y un `ColumnTransformer` que aplica cada uno a **sus** columnas.

In [ ]:
datos = viv.dropna(subset=["estado"]).copy()

cat_onehot = Pipeline(steps=[("encoder", OneHotEncoder(handle_unknown="ignore"))])
cat_ordinal = Pipeline(steps=[("encoder", OrdinalEncoder())])
num = Pipeline(steps=[("transf", PowerTransformer())])

preprocessor = ColumnTransformer([
    ("onehot", cat_onehot, ["tipo_inmueble", "pais"]),
    ("ordinal", cat_ordinal, ["estado", "distrito"]),
    ("num", num, ["area_m2", "habitaciones"]),
], sparse_threshold=0)

pipeline = Pipeline(steps=[("preprocessor", preprocessor)])
X = pipeline.fit_transform(datos)
salida = pd.DataFrame(X, columns=pipeline.get_feature_names_out())
print(salida.shape)
salida.head()

Cambiar **un** transformador (por ejemplo `StandardScaler` en lugar de `PowerTransformer`) no toca las demás columnas:

In [ ]:
preprocessor.set_params(num=Pipeline([("scaler", StandardScaler())]))
X2 = pipeline.fit_transform(datos)
print("¿Categóricas iguales?", np.array_equal(X[:, :-2], X2[:, :-2]))
print("Media / std de las numéricas:", X2[:, -2:].mean(axis=0).round(3), X2[:, -2:].std(axis=0).round(3))

## ✏️ Tu turno 1
Cuenta cuántas columnas saldrían si codificas `distrito`, `tipo_inmueble`, `estado` y `pais` con OneHot, con dummy y con binario.
¿Cuál elegirías para `distrito`? ¿Por qué?

In [ ]:
# Tu código aquí

## ✏️ Tu turno 2
Aplica `MinMaxScaler` y `PowerTransformer` a `area_m2` (tiene outliers ×10). Compara el `skew` y el máximo.
¿Cambió la forma con MinMax? ¿Y con PowerTransformer?

In [ ]:
# Tu código aquí

## ✏️ Tu turno 3
Arma un `ColumnTransformer` para `clientes_banco.csv`: `segmento` con OneHot y `edad`, `ingreso_mensual`, `antiguedad_anios`
con `StandardScaler`. ¿Cuántas columnas salen? Revisa los nombres con `get_feature_names_out()`.

In [ ]:
# Tu código aquí